# Получение YT_REFRESH_TOKEN (один раз)

Этот блокнот нужен, чтобы робот мог загружать видео на ваш YouTube-канал.

**Что нужно заранее:** `Client ID` и `Client Secret` из Google Cloud (тип приложения **«Компьютерное приложение / Desktop app»**). Как их получить — в README репозитория.

**Как пользоваться (с телефона):**
1. Нажмите ▶ у **Шага 1** — установятся библиотеки.
2. В **Шаге 2** вставьте Client ID и Client Secret в поля справа/снизу и нажмите ▶. Появится ссылка.
3. Откройте ссылку, войдите в аккаунт Google **того канала**, куда будут грузиться видео, и разрешите доступ.
   Если появится «Приложение не проверено» → «Дополнительно» → «Перейти на страницу …».
4. После разрешения браузер откроет страницу `localhost`, которая **не загрузится** — это нормально!
   Скопируйте **полный адрес** этой страницы из адресной строки (он начинается с `http://localhost`).
5. Нажмите ▶ у **Шага 3** и вставьте скопированный адрес в поле. Получите токен.

## Шаг 1 — установка

In [ ]:
!pip -q install google-auth-oauthlib

## Шаг 2 — ссылка для входа

In [ ]:
#@title Вставьте ключи и нажмите ▶
CLIENT_ID = "" #@param {type:"string"}
CLIENT_SECRET = "" #@param {type:"string"}

import os
os.environ["OAUTHLIB_RELAX_TOKEN_SCOPE"] = "1"
from google_auth_oauthlib.flow import Flow

CLIENT_ID = CLIENT_ID.strip()
CLIENT_SECRET = CLIENT_SECRET.strip()
assert CLIENT_ID and CLIENT_SECRET, "Сначала вставьте CLIENT_ID и CLIENT_SECRET в поля выше"

config = {"installed": {
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "auth_uri": "https://accounts.google.com/o/oauth2/auth",
    "token_uri": "https://oauth2.googleapis.com/token",
    "redirect_uris": ["http://localhost"],
}}
flow = Flow.from_client_config(
    config,
    scopes=["https://www.googleapis.com/auth/youtube.upload"],
    redirect_uri="http://localhost:8080/",
)
auth_url, _ = flow.authorization_url(access_type="offline", prompt="consent")
print("👉 Откройте эту ссылку, войдите и разрешите доступ:\n")
print(auth_url)

## Шаг 3 — получить токен

In [ ]:
#@title Вставьте адрес страницы localhost и нажмите ▶
REDIRECTED_URL = "" #@param {type:"string"}

from urllib.parse import urlparse, parse_qs
url = REDIRECTED_URL.strip()
assert url, "Вставьте полный адрес страницы (начинается с http://localhost)"
if url.startswith("http"):
    q = parse_qs(urlparse(url).query)
    if "error" in q:
        raise SystemExit("Google вернул ошибку: " + q["error"][0])
    code = q["code"][0]
else:
    code = url  # вставили только сам код
flow.fetch_token(code=code)
token = flow.credentials.refresh_token
if not token:
    print("❌ Refresh token не выдан. Удалите доступ приложения на https://myaccount.google.com/permissions и повторите Шаг 2 и 3.")
else:
    print("✅ Готово! Скопируйте строку ниже целиком и сохраните её в GitHub как секрет YT_REFRESH_TOKEN:\n")
    print(token)

**Важно:** никому не показывайте этот токен. Если ошиблись — просто запустите Шаг 2 и Шаг 3 заново (ссылка из Шага 2 одноразовая, после повторного запуска Шага 2 нужна новая).